In [ ]:
import requests
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta

In [ ]:
test_request = requests.get("https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/acceptance/volumes/all/bid/2024-02-01")

In [ ]:
test_request.request

<PreparedRequest [GET]>


In [ ]:
test_request_dict = test_request.json()

In [ ]:
df = pd.json_normalize(test_request_dict["data"])


In [ ]:
## Just keeping this in for reference. Did something else in the end.

"""
What I need to do:
Download:
# Acceptance volumes by settlement date (BOAV)
Bid dataset, offer data set

With the following variable:
BmUnit - The specific generating unit
leadPartyName - The company or asset name responsible for the BMU
bmUnitTypeT (Transmission-Connected) or E (Embedded).Confirms if it is a Centralized unit, making its volume acceptance directly relevant to BM actions
totalVolumeAccepted - The Net Volume Accepted. The total volume of Bids and Offers accepted for this BMU in the period. The negative sign indicates an acceptance of a Bid (a reduction in generation or increase in consumption).
acceptanceId

Step 1: Download the data
### Write a function that takes date in format "yyyy-MM-dd", and a string of either "bid" or "offer", and returns the dataframe of the data

Step 2: format the data
Delete all columns that aren't BmUnit, leadPartyName, bmUnitTypeT, totalVolumeAccepted

Format the time of each row into yyyy-mm-dd-sp

"""


Redo it, basing it off of Zuo's code. Better!

In [ ]:
## OLD. THIS COPY OF THE CODE DOESN'T FORMAT IT AS NEEDED.


## THIS IS FOR THE BID ACCEPTANCE VOLUMES. To get the offer acceptance volumes
## Copy change 'bid' to 'offer in BASE_URL, and change any files or column names from
## 'NBAV' to 'NAOV'.

import requests
import time
import pandas as pd
from datetime import datetime, timedelta
import os
import sys
import json

# --- Configuration ---
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/acceptance/volumes/all/bid/" # data goes on the end in the format yyyy-MM-dd

START_DATE = datetime(2021, 4, 1)
END_DATE = datetime(2025, 4, 1)
SP_FROM = 1
SP_TO = 48

TIME_DELAY_SECONDS = 5  # Recommended delay to avoid rate limiting
REQUEST_TIMEOUT_SECONDS = 30 # Increased timeout for slow API responses
OUTPUT_FILE_NAME = 'elexon_BAV_data_2021_2025.csv'

total_records_retrieved = 0

# --- Function to Append Records to CSV ---
def append_to_csv(data_records, file_name, write_header=False):
    """Converts a list of dicts to a DataFrame and appends to a CSV file."""
    if not data_records:
        return 0

    try:
        df = pd.DataFrame(data_records)

        # Use 'a' (append) mode. Only write header if the file doesn't exist yet,
        # or if explicitly told to (when initializing).
        header = write_header and not os.path.exists(file_name)

        df.to_csv(file_name, mode='a', header=header, index=False)
        return len(df)
    except Exception as e:
        print(f"-> ❌ ERROR: Failed to convert or save data to CSV. {e}")
        return 0

# --- Data Retrieval Loop ---

# 1. Initialize the current date tracker
current_date = START_DATE
print(f"Starting data retrieval from {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}...")

while current_date <= END_DATE:



    """ I don't think I need this bit from Zuo's code as for BOAV, the date for the query is passed as part of the URL, not as a standard query parameter.
    params = {
        'from': from_date,
        'to': to_date,
        'settlementPeriodFrom': SP_FROM,
        'settlementPeriodTo': SP_TO,
        'format': 'json'
    }
    """

    date_str = current_date.strftime('%Y-%m-%d')
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Request: Querying {date_str} (all settlement periods (1 to 46-50)...", end="")

    try:
        # 2. Execute the request with an increased timeout
        response = requests.get(f"{BASE_URL}{current_date.strftime('%Y-%m-%d')}", timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status() # Check for 4xx/5xx status codes

        # 3. Extract records from response JSON
        data = response.json()

        # Handle nested data structure
        records = []
        if isinstance(data, dict) and 'data' in data:
            records = data['data']
        elif isinstance(data, list):
            records = data

        # 4. Save the records immediately to the CSV file
        # The header is written only on the first *successful* data retrieval.
        is_first_write = total_records_retrieved == 0
        records_saved = append_to_csv(records, OUTPUT_FILE_NAME, write_header=is_first_write)

        total_records_retrieved += records_saved
        print(f" -> ✅ Success: Saved {records_saved} records. Total: {total_records_retrieved}")

    except requests.exceptions.Timeout:
        print(f" -> ❌ **Timeout Error**: Took longer than {REQUEST_TIMEOUT_SECONDS}s.")
    except requests.exceptions.HTTPError as e:
        print(f" -> ❌ **HTTP Error**: {e.response.status_code}. Details: {e.response.text.strip()}")
    except requests.exceptions.RequestException as e:
        print(f" -> ❌ **Connection Error**: {e}")
    except json.JSONDecodeError:
        print(f" -> ❌ **JSON Error**: Failed to decode JSON response.")
    except Exception as e:
        print(f" -> ❌ **Unexpected Error**: {e}")

    # 5. Advance the date
    current_date += timedelta(days=1)

    # 6. Apply Time Delay (CRITICAL STEP)
    if current_date <= END_DATE:
        sys.stdout.flush() # Ensure print statements appear before the sleep
        print(f"Waiting {TIME_DELAY_SECONDS} seconds...")
        time.sleep(TIME_DELAY_SECONDS)

print("\n--- Data retrieval complete ---")
print(f"Final data saved to: {OUTPUT_FILE_NAME}")
print(f"Total successful records retrieved and saved: {total_records_retrieved}")

Starting data retrieval from 2021-04-01 to 2025-04-01...
[16:28:49] Request: Querying 2021-04-01 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 2655 records. Total: 2655
Waiting 5 seconds...
[16:28:55] Request: Querying 2021-04-02 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 2173 records. Total: 4828
Waiting 5 seconds...
[16:29:01] Request: Querying 2021-04-03 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 2433 records. Total: 7261
Waiting 5 seconds...
[16:29:07] Request: Querying 2021-04-04 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 3872 records. Total: 11133
Waiting 5 seconds...
[16:29:13] Request: Querying 2021-04-05 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 3013 records. Total: 14146
Waiting 5 seconds...
[16:29:19] Request: Querying 2021-04-06 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 2695 records. Total: 16841
Waiting 5 seconds...
[16:29:24] Request: Querying 2021-04-07 (all settlement pe

In [ ]:
#mount gdrive to load data on gdrive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


**First**, you need to identify the exact path to your shared drive folder after it's been mounted. You can list the contents of your Google Drive to find it.

In [ ]:
import pandas as pd

# Assuming the 'all_pn_data' from the previous cell is the DataFrame you want to save.
# If you have another DataFrame, replace 'all_pn_data' with its name.

# IMPORTANT: Replace 'Your_Shared_Drive_Name_Or_Path' with the actual path to your shared drive folder.
# Example: SHARED_DRIVE_PATH = '/content/drive/Shareddrives/My_Group_Shared_Drive/my_data_folder/'
SHARED_DRIVE_PATH = '/content/drive/MyDrive/Data Analysis Group Coursework/data/'

# Ensure the directory exists (optional, but good practice)
os.makedirs(SHARED_DRIVE_PATH, exist_ok=True)

output_file_name_in_drive = os.path.join(SHARED_DRIVE_PATH, 'elexon_NBAV_data_2021_2025.csv')

print(f"the csv is found in: {output_file_name_in_drive}")

the csv is found in: /content/drive/MyDrive/Data Analysis Group Coursework/data/elexon_BOV_data_2021_2025.csv


Pause. I'm redoing the function so that it formats it automatically each time it appends the next day to the csv file. This has taken me fucking ages to do (I'm going to guess three hours lol). BUT it will be so satisfying if it actually works. Fingers crossed.

In [ ]:
## THIS IS FOR THE BID ACCEPTANCE VOLUMES. To get the offer acceptance volumes
## Copy change 'bid' to 'offer in BASE_URL, and change any files or column names from
## 'NBAV' to 'NAOV'.

import requests
import time
import pandas as pd
from datetime import datetime, timedelta
import os
import sys
import json

# --- Configuration ---
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/acceptance/volumes/all/bid/" # data goes on the end in the format yyyy-MM-dd

START_DATE = datetime(2021, 4, 1)
END_DATE = datetime(2025, 4, 1)
SP_FROM = 1
SP_TO = 48

TIME_DELAY_SECONDS = 5  # Recommended delay to avoid rate limiting
REQUEST_TIMEOUT_SECONDS = 30 # Increased timeout for slow API responses
OUTPUT_FILE_NAME = 'elexon_NBAV_data_2021_2025.csv'

total_records_retrieved = 0

# --- Function to Append Records to CSV ---
def append_to_csv(data_records, file_name, write_header=False):
    """Converts a list of dicts to a DataFrame and appends to a CSV file."""
    if not data_records:
        return 0

    try:
        df = pd.DataFrame(data_records)

        ## First, make sure the settlementPeriod column is correctly formatted.
        df["settlementPeriod"] = df["settlementPeriod"].apply(lambda col: str(col).zfill(2))

        ## Second, add a correctly formatted date_sp column, with the format yyyy-MM-dd-sp
        df["date_sp"] = df["settlementDate"].str.cat(df["settlementPeriod"].astype(str), sep ="-")

        ## Finally, use the ugliest pandas function in the world to produce a DataFrame
        ## that has been one row for each date_sp, with one value for
        ## volume accepted, along with a proportion for short periods vs long
        formatted_df = df.groupby("date_sp").agg(
            NBAV_total_accepted_volume_per_period = ("totalVolumeAccepted","sum"),
            NBAV_proportion_of_short_period_volumes_to_long=('acceptanceDuration', lambda x: (x == 'S').mean())
            ).reset_index().sort_values(by = 'date_sp',ascending = True)


        # Use 'a' (append) mode. Only write header if the file doesn't exist yet,
        # or if explicitly told to (when initializing).
        header = write_header and not os.path.exists(file_name)

        formatted_df.to_csv(file_name, mode='a', header=header, index=False)
        return len(formatted_df)
    except Exception as e:
        print(f"-> ❌ ERROR: Failed to convert or save data to CSV. {e}")
        return 0

# --- Data Retrieval Loop ---

# 1. Initialize the current date tracker
current_date = START_DATE
print(f"Starting data retrieval from {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}...")

while current_date <= END_DATE:

    date_str = current_date.strftime('%Y-%m-%d')
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Request: Querying {date_str} (all settlement periods (1 to 46-50)...", end="")

    try:
        # 2. Execute the request with an increased timeout
        response = requests.get(f"{BASE_URL}{current_date.strftime('%Y-%m-%d')}", timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status() # Check for 4xx/5xx status codes

        # 3. Extract records from response JSON
        data = response.json()

        # Handle nested data structure
        records = []
        if isinstance(data, dict) and 'data' in data:
            records = data['data']
        elif isinstance(data, list):
            records = data

        # 4. Save the records immediately to the CSV file
        # The header is written only on the first *successful* data retrieval.
        is_first_write = total_records_retrieved == 0
        records_saved = append_to_csv(records, OUTPUT_FILE_NAME, write_header=is_first_write)

        total_records_retrieved += records_saved
        print(f" -> ✅ Success: Saved {records_saved} records. Total: {total_records_retrieved}")

    except requests.exceptions.Timeout:
        print(f" -> ❌ **Timeout Error**: Took longer than {REQUEST_TIMEOUT_SECONDS}s.")
    except requests.exceptions.HTTPError as e:
        print(f" -> ❌ **HTTP Error**: {e.response.status_code}. Details: {e.response.text.strip()}")
    except requests.exceptions.RequestException as e:
        print(f" -> ❌ **Connection Error**: {e}")
    except json.JSONDecodeError:
        print(f" -> ❌ **JSON Error**: Failed to decode JSON response.")
    except Exception as e:
        print(f" -> ❌ **Unexpected Error**: {e}")

    # 5. Advance the date
    current_date += timedelta(days=1)

    # 6. Apply Time Delay (CRITICAL STEP)
    if current_date <= END_DATE:
        sys.stdout.flush() # Ensure print statements appear before the sleep
        print(f"Waiting {TIME_DELAY_SECONDS} seconds...")
        time.sleep(TIME_DELAY_SECONDS)

print("\n--- Data retrieval complete ---")
print(f"Final data saved to: {OUTPUT_FILE_NAME}")
print(f"Total successful records retrieved and saved: {total_records_retrieved}")

Starting data retrieval from 2021-04-01 to 2025-04-01...
[19:35:50] Request: Querying 2021-04-01 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 48
Waiting 5 seconds...
[19:35:56] Request: Querying 2021-04-02 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 96
Waiting 5 seconds...
[19:36:01] Request: Querying 2021-04-03 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 144
Waiting 5 seconds...
[19:36:07] Request: Querying 2021-04-04 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 192
Waiting 5 seconds...
[19:36:13] Request: Querying 2021-04-05 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 240
Waiting 5 seconds...
[19:36:19] Request: Querying 2021-04-06 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 288
Waiting 5 seconds...
[19:36:25] Request: Querying 2021-04-07 (all settlement periods (1 to 46-50)... -

In [ ]:
#mount gdrive to load data on gdrive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# Assuming the 'all_pn_data' from the previous cell is the DataFrame you want to save.
# If you have another DataFrame, replace 'all_pn_data' with its name.

# IMPORTANT: Replace 'Your_Shared_Drive_Name_Or_Path' with the actual path to your shared drive folder.
# Example: SHARED_DRIVE_PATH = '/content/drive/Shareddrives/My_Group_Shared_Drive/my_data_folder/'
SHARED_DRIVE_PATH = '/drive/MyDrive/Energy Data Analysis Group 3 Coursework/data/Raw'

# Ensure the directory exists (optional, but good practice)
os.makedirs(SHARED_DRIVE_PATH, exist_ok=True)

output_file_name_in_drive = os.path.join(SHARED_DRIVE_PATH, 'elexon_NBAV_data_2021_2025.csv')

print(f"the csv is found in: {output_file_name_in_drive}")

the csv is found in: /drive/MyDrive/Energy Data Analysis Group 3 Coursework/data/Raw/elexon_NBAV_data_2021_2025.csv


In [ ]:
os.getcwd()

'/content'

In [ ]:
## THIS IS FOR THE OFFER ACCEPTANCE VOLUMES. blablablabla, look at the code above for the other one dickhead.

import requests
import time
import pandas as pd
from datetime import datetime, timedelta
import os
import sys
import json

# --- Configuration ---
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/acceptance/volumes/all/offer/" # data goes on the end in the format yyyy-MM-dd

START_DATE = datetime(2021, 4, 1)
END_DATE = datetime(2025, 4, 1)
SP_FROM = 1
SP_TO = 48

TIME_DELAY_SECONDS = 5  # Recommended delay to avoid rate limiting
REQUEST_TIMEOUT_SECONDS = 30 # Increased timeout for slow API responses

## OUTPUT STRAIGHT TO GOOGLE DRIVE
OUTPUT_FILE_NAME = '/content/drive/MyDrive/Energy Data Analysis Group 3 Coursework/data/Raw/elexon_NOAV_data_2021_2025.csv'

total_records_retrieved = 0

# --- Function to Append Records to CSV ---
def append_to_csv(data_records, file_name, write_header=False):
    """Converts a list of dicts to a DataFrame and appends to a CSV file."""
    if not data_records:
        return 0

    try:
        df = pd.DataFrame(data_records)

        ## First, make sure the settlementPeriod column is correctly formatted.
        df["settlementPeriod"] = df["settlementPeriod"].apply(lambda col: str(col).zfill(2))

        ## Second, add a correctly formatted date_sp column, with the format yyyy-MM-dd-sp
        df["date_sp"] = df["settlementDate"].str.cat(df["settlementPeriod"].astype(str), sep ="-")

        ## Finally, use the ugliest pandas function in the world to produce a DataFrame
        ## that has been one row for each date_sp, with one value for
        ## volume accepted, along with a proportion for short periods vs long
        formatted_df = df.groupby("date_sp").agg(
            NOAV_total_accepted_volume_per_period = ("totalVolumeAccepted","sum"),
            NOAV_proportion_of_short_period_volumes_to_long=('acceptanceDuration', lambda x: (x == 'S').mean())
            ).reset_index().sort_values(by = 'date_sp',ascending = True)


        # Use 'a' (append) mode. Only write header if the file doesn't exist yet,
        # or if explicitly told to (when initializing).
        header = write_header and not os.path.exists(file_name)

        formatted_df.to_csv(file_name, mode='a', header=header, index=False)
        return len(formatted_df)
    except Exception as e:
        print(f"-> ❌ ERROR: Failed to convert or save data to CSV. {e}")
        return 0

# --- Data Retrieval Loop ---

# 1. Initialize the current date tracker
current_date = START_DATE
print(f"Starting data retrieval from {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}...")

while current_date <= END_DATE:

    date_str = current_date.strftime('%Y-%m-%d')
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Request: Querying {date_str} (all settlement periods (1 to 46-50)...", end="")

    try:
        # 2. Execute the request with an increased timeout
        response = requests.get(f"{BASE_URL}{current_date.strftime('%Y-%m-%d')}", timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status() # Check for 4xx/5xx status codes

        # 3. Extract records from response JSON
        data = response.json()

        # Handle nested data structure
        records = []
        if isinstance(data, dict) and 'data' in data:
            records = data['data']
        elif isinstance(data, list):
            records = data

        # 4. Save the records immediately to the CSV file
        # The header is written only on the first *successful* data retrieval.
        is_first_write = total_records_retrieved == 0
        records_saved = append_to_csv(records, OUTPUT_FILE_NAME, write_header=is_first_write)

        total_records_retrieved += records_saved
        print(f" -> ✅ Success: Saved {records_saved} records. Total: {total_records_retrieved}")

    except requests.exceptions.Timeout:
        print(f" -> ❌ **Timeout Error**: Took longer than {REQUEST_TIMEOUT_SECONDS}s.")
    except requests.exceptions.HTTPError as e:
        print(f" -> ❌ **HTTP Error**: {e.response.status_code}. Details: {e.response.text.strip()}")
    except requests.exceptions.RequestException as e:
        print(f" -> ❌ **Connection Error**: {e}")
    except json.JSONDecodeError:
        print(f" -> ❌ **JSON Error**: Failed to decode JSON response.")
    except Exception as e:
        print(f" -> ❌ **Unexpected Error**: {e}")

    # 5. Advance the date
    current_date += timedelta(days=1)

    # 6. Apply Time Delay (CRITICAL STEP)
    if current_date <= END_DATE:
        sys.stdout.flush() # Ensure print statements appear before the sleep
        print(f"Waiting {TIME_DELAY_SECONDS} seconds...")
        time.sleep(TIME_DELAY_SECONDS)

print("\n--- Data retrieval complete ---")
print(f"Final data saved to: {OUTPUT_FILE_NAME}")
print(f"Total successful records retrieved and saved: {total_records_retrieved}")

Starting data retrieval from 2021-04-01 to 2025-04-01...
[22:50:54] Request: Querying 2021-04-01 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 48
Waiting 5 seconds...
[22:51:00] Request: Querying 2021-04-02 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 96
Waiting 5 seconds...
[22:51:06] Request: Querying 2021-04-03 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 144
Waiting 5 seconds...
[22:51:13] Request: Querying 2021-04-04 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 192
Waiting 5 seconds...
[22:51:19] Request: Querying 2021-04-05 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 240
Waiting 5 seconds...
[22:51:25] Request: Querying 2021-04-06 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 288
Waiting 5 seconds...
[22:51:31] Request: Querying 2021-04-07 (all settlement periods (1 to 46-50)... -

In [ ]:
## THIS IS FOR SYSTEM PRICE

import requests
import time
import pandas as pd
from datetime import datetime, timedelta
import os
import sys
import json

# --- Configuration ---
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/system-prices/" # data goes on the end in the format yyyy-MM-dd

START_DATE = datetime(2021, 4, 1)
END_DATE = datetime(2025, 4, 1)
SP_FROM = 1
SP_TO = 48

TIME_DELAY_SECONDS = 5  # Recommended delay to avoid rate limiting
REQUEST_TIMEOUT_SECONDS = 30 # Increased timeout for slow API responses

## OUTPUT STRAIGHT TO GOOGLE DRIVE
OUTPUT_FILE_NAME = '/content/drive/MyDrive/Energy Data Analysis Group 3 Coursework/data/Raw/elexon_system_price_data_2021_2025.csv'

total_records_retrieved = 0

# --- Function to Append Records to CSV ---
def append_to_csv(data_records, file_name, write_header=False):
    """Converts a list of dicts to a DataFrame and appends to a CSV file."""
    if not data_records:
        return 0

    try:
        df = pd.DataFrame(data_records)

        ## First, make sure the settlementPeriod column is correctly formatted.
        df["settlementPeriod"] = df["settlementPeriod"].apply(lambda col: str(col).zfill(2))

        ## Second, add a correctly formatted date_sp column, with the format yyyy-MM-dd-sp
        df["date_sp"] = df["settlementDate"].str.cat(df["settlementPeriod"].astype(str), sep ="-")

        ## Simpler function than the last ones.
        formatted_df = df[["date_sp", "systemSellPrice","systemBuyPrice","netImbalanceVolume"]].sort_values(by = 'date_sp',ascending = True)

        # Use 'a' (append) mode. Only write header if the file doesn't exist yet,
        # or if explicitly told to (when initializing).
        header = write_header and not os.path.exists(file_name)

        formatted_df.to_csv(file_name, mode='a', header=header, index=False)
        return len(formatted_df)
    except Exception as e:
        print(f"-> ❌ ERROR: Failed to convert or save data to CSV. {e}")
        return 0

# --- Data Retrieval Loop ---

# 1. Initialize the current date tracker
current_date = START_DATE
print(f"Starting data retrieval from {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}...")

while current_date <= END_DATE:

    date_str = current_date.strftime('%Y-%m-%d')
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Request: Querying {date_str} (all settlement periods (1 to 46-50)...", end="")

    try:
        # 2. Execute the request with an increased timeout
        response = requests.get(f"{BASE_URL}{current_date.strftime('%Y-%m-%d')}?format=json", timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status() # Check for 4xx/5xx status codes

        # 3. Extract records from response JSON
        data = response.json()

        # Handle nested data structure
        records = []
        if isinstance(data, dict) and 'data' in data:
            records = data['data']
        elif isinstance(data, list):
            records = data

        # 4. Save the records immediately to the CSV file
        # The header is written only on the first *successful* data retrieval.
        is_first_write = total_records_retrieved == 0
        records_saved = append_to_csv(records, OUTPUT_FILE_NAME, write_header=is_first_write)

        total_records_retrieved += records_saved
        print(f" -> ✅ Success: Saved {records_saved} records. Total: {total_records_retrieved}")

    except requests.exceptions.Timeout:
        print(f" -> ❌ **Timeout Error**: Took longer than {REQUEST_TIMEOUT_SECONDS}s.")
    except requests.exceptions.HTTPError as e:
        print(f" -> ❌ **HTTP Error**: {e.response.status_code}. Details: {e.response.text.strip()}")
    except requests.exceptions.RequestException as e:
        print(f" -> ❌ **Connection Error**: {e}")
    except json.JSONDecodeError:
        print(f" -> ❌ **JSON Error**: Failed to decode JSON response.")
    except Exception as e:
        print(f" -> ❌ **Unexpected Error**: {e}")

    # 5. Advance the date
    current_date += timedelta(days=1)

    # 6. Apply Time Delay (CRITICAL STEP)
    if current_date <= END_DATE:
        sys.stdout.flush() # Ensure print statements appear before the sleep
        print(f"Waiting {TIME_DELAY_SECONDS} seconds...")
        time.sleep(TIME_DELAY_SECONDS)

print("\n--- Data retrieval complete ---")
print(f"Final data saved to: {OUTPUT_FILE_NAME}")
print(f"Total successful records retrieved and saved: {total_records_retrieved}")

Starting data retrieval from 2021-04-01 to 2025-04-01...
[11:37:17] Request: Querying 2021-04-01 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 48
Waiting 5 seconds...
[11:37:23] Request: Querying 2021-04-02 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 96
Waiting 5 seconds...
[11:37:28] Request: Querying 2021-04-03 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 144
Waiting 5 seconds...
[11:37:34] Request: Querying 2021-04-04 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 192
Waiting 5 seconds...
[11:37:39] Request: Querying 2021-04-05 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 240
Waiting 5 seconds...
[11:37:44] Request: Querying 2021-04-06 (all settlement periods (1 to 46-50)... -> ✅ Success: Saved 48 records. Total: 288
Waiting 5 seconds...
[11:37:49] Request: Querying 2021-04-07 (all settlement periods (1 to 46-50)... -